# Lab 4: Retrieval‑Augmented Generation (RAG) Implementation

In this lab you will build a simple RAG pipeline that:
1. Indexes a small collection of documents.
2. Retrieves the most relevant passages for a user query.
3. Constructs a prompt that includes the retrieved passages and a system prompt.
4. Calls an LLM to generate a final answer.

The goal is to see how external knowledge can be injected into the model to improve factuality and reduce hallucinations.


In [ ]:
import os
import openai
from sentence_transformers import SentenceTransformer
import faiss

# Set your OpenAI API key
openai.api_key = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY")

# 1️⃣  Sample corpus – replace with your own documents if desired
corpus = [
    "FastAPI позволяет объявлять зависимости через Depends().",
    "SQLAlchemy 2.0 поддерживает асинхронные сессии через async_session.",
    "Docker упрощает развертывание микросервисов в контейнерах.",
    "PostgreSQL поддерживает расширения, такие как pg_trgm для ускорения поиска.",
    "Redis часто используется как кэш для ускорения чтения данных."
]

# 2️⃣  Encode documents using a dense encoder (same tokenizer as the target LLM)
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(corpus, convert_to_numpy=True)

# 3️⃣  Build a FAISS index for fast similarity search
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

def retrieve(query: str, k: int = 3):
    """Return the top‑k most similar passages from the corpus."""
    q_emb = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(q_emb, k)
    return [corpus[i] for i in indices[0]]

def generate_answer(query: str):
    passages = retrieve(query)
    system_prompt = "Ты — senior backend‑разработчик. Используй приведённые ниже фрагменты как авторитетный источник."
    retrieved_text = "\n".join([f"{i+1}. {p}" for i, p in enumerate(passages)])
    full_prompt = (
        f"{system_prompt}\n"
        f"Retrieved passages:\n{retrieved_text}\n\n"
        f"User: {query}"
    )
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": full_prompt}],
        temperature=0,
        max_tokens=300,
    )
    return response["choices"][0]["message"]["content"].strip()

# Example query – run this cell after the definitions above
example_query = "Как реализовать асинхронный доступ к базе в FastAPI?"
answer = generate_answer(example_query)
print(answer)


## Reflection

- How many passages were retrieved? Try changing `k` in `retrieve()`.
- Does the answer reference the retrieved information?
- Experiment with a different system prompt (e.g., ask for a concise bullet list).
